## 1- Data Analysis 

1- Check image file types

2- Dataset inventory tables

    -  Image count, mask count, matched pairs, missing items (ISIC + HAM)
3- Image and mask statistics

    - Resolution distribution (H, W)

    - Lesion area distribution (mask coverage %)

4- Class distribution (HAM10000)

    - Counts per dx (and optionally grouped malignant vs benign)

5- small “edge-case” gallery

    - Smallest lesions, largest lesions, weird aspect ratios, low contrast examples
    

### 1.1 - Data directory
Dataset Storage (Local Only)

This folder contains datasets used for the **DermaXplain** practicum project.
**All contents of this directory are excluded from version control** and must be downloaded locally by each collaborator.

### Directory Structure initial
```
data/
|--- isic2018/
│    |--- images/
│    |--- masks/
|--- ham10000/
│    |--- images/
|    |--- masks/
│    |--- metadata.csv
```

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

# Adjust if this notebook is not in <dir>/
ROOT = Path("..")
ISIC_DIR = ROOT / "data" / "isic2018"
ISIC_IMG_DIR  = ISIC_DIR / "images"
ISIC_MASK_DIR = ISIC_DIR / "masks"

HAM_DIR = ROOT / "data" / "ham10000"
HAM_IMG_DIR = HAM_DIR / "images"
HAM_META = HAM_DIR / "HAM10000_metadata.csv"
HAM_MASK_DIR = HAM_DIR / "masks"  # Tschandl masks unzipped

assert ISIC_IMG_DIR.exists(), f"Missing {ISIC_IMG_DIR}"
assert ISIC_MASK_DIR.exists(), f"Missing {ISIC_MASK_DIR}"


In [ ]:
import sys, torch
print(sys.executable)
print(torch.__version__)
print(torch.cuda.is_available())


In [ ]:
from collections import Counter

def suffix_count(dir_path: Path):
    files = [p for p in dir_path.iterdir() if p.is_file()]
    return Counter(p.suffix.lower() for p in files)

print("ISIC images:", suffix_count(ISIC_IMG_DIR))
print("ISIC masks :", suffix_count(ISIC_MASK_DIR))

print("HAM images :", suffix_count(HAM_IMG_DIR))
print("HAM masks  :", suffix_count(HAM_MASK_DIR))

### 1.2 - Data directory new structure
```
data/isic2018/
|--  images/        # .jpg only
|--  masks/         # .png only
|--  txt/
  |--  images/      # .txt originally from images/
  |--  masks/       # .txt originally from masks/


data/ham10000/
|--  images/        # .jpg only
|--  masks/         # .png only
|--  txt/
  |--  images/      # .txt originally from images/
  |--  masks/       # .txt originally from masks/
```

In [ ]:
# moving txt files to a separate folder
import shutil

def move_txt_files(base_dir: Path):
    moved = []

    for sub in ["images", "masks"]:
        src_dir = base_dir / sub
        if not src_dir.exists():
            continue

        dest_dir = base_dir / "txt" / sub
        dest_dir.mkdir(parents=True, exist_ok=True)

        for p in src_dir.iterdir():
            if p.is_file() and p.suffix.lower() == ".txt":
                dest = dest_dir / p.name
                if not dest.exists():
                    shutil.move(str(p), str(dest))
                    moved.append(str(dest))

    return moved


# ISIC
moved_isic = move_txt_files(ISIC_DIR)
print(f"ISIC moved {len(moved_isic)} txt files")

# HAM (for symmetry / future-proofing)
moved_ham = move_txt_files(HAM_DIR)
print(f"HAM moved {len(moved_ham)} txt files")

assert not any(p.suffix == ".txt" for p in ISIC_IMG_DIR.iterdir())
assert not any(p.suffix == ".txt" for p in ISIC_MASK_DIR.iterdir())
assert not any(p.suffix == ".txt" for p in HAM_IMG_DIR.iterdir())
assert not any(p.suffix == ".txt" for p in HAM_MASK_DIR.iterdir())


## 2 - Files renaming and inventory
### 2.1- Remove the string '_segmentation' from suffix in masks
This will simplify the scripts with Uniform mapping: image ISIC_xxx.jpg <--> mask ISIC_xxx.png

In [ ]:
# Rename mask files from <stem>_segmentation.png - > <stem>.png
def remove_segmentation_suffix(mask_dir: Path, dry_run: bool = False) -> list[tuple[str, str]]:
    """
    Rename mask files from <stem>_segmentation.png → <stem>.png
    Returns a list of (old_name, new_name).
    """
    renames = []

    for p in mask_dir.iterdir():
        if not p.is_file():
            continue
        if p.suffix.lower() != ".png":
            continue
        if not p.stem.endswith("_segmentation"):
            continue

        new_name = p.stem.removesuffix("_segmentation") + ".png"
        new_path = p.with_name(new_name)

        if new_path.exists():
            raise FileExistsError(f"Collision detected: {new_path}")

        renames.append((p.name, new_name))

        if not dry_run:
            p.rename(new_path)

    return renames

#  Rename mask files from <stem>.png -> <stem>_segmentation.png

def add_segmentation_suffix(mask_dir: Path, dry_run: bool = False) -> list[tuple[str, str]]:
    """
    Rename mask files from <stem>.png -> <stem>_segmentation.png
    Returns a list of (old_name, new_name).
    """
    renames = []

    for p in mask_dir.iterdir():
        if not p.is_file():
            continue
        if p.suffix.lower() != ".png":
            continue
        if p.stem.endswith("_segmentation"):
            continue

        new_name = p.stem + "_segmentation.png"
        new_path = p.with_name(new_name)

        if new_path.exists():
            raise FileExistsError(f"Collision detected: {new_path}")

        renames.append((p.name, new_name))

        if not dry_run:
            p.rename(new_path)

    return renames



## Remove segmentation suffixes in ISIC masks
renamed_isic = remove_segmentation_suffix(ISIC_MASK_DIR)
print(f"ISIC renamed {len(renamed_isic)} mask files")     

## Remove segmentation suffixes in HAM masks
renamed_ham = remove_segmentation_suffix(HAM_MASK_DIR)
print(f"HAM renamed {len(renamed_ham)} mask files")    

## to revert changes: uncomment the following lines:
'''
## Add segmentation suffixes back in ISIC masks
renamed_isic_back = add_segmentation_suffix(ISIC_MASK_DIR)
print(f"ISIC renamed back {len(renamed_isic_back)} mask files")   

## Add segmentation suffixes back in HAM masks
renamed_ham_back = add_segmentation_suffix(HAM_MASK_DIR)
print(f"HAM renamed back {len(renamed_ham_back)} mask files")   
'''

## 2.2 - Inventory and matching

In [ ]:
# Helpers

def list_files(dir_path: Path, suffix: str) -> list[Path]:
    return sorted([p for p in dir_path.iterdir() if p.is_file() and p.suffix.lower() == suffix])

def stems(dir_path: Path, suffix: str) -> set[str]:
    return {p.stem for p in list_files(dir_path, suffix)}

def load_rgb_u8(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("RGB"), dtype=np.uint8)

def load_mask_bin(path: Path) -> np.ndarray:
    # binary mask (0/1)
    m = np.array(Image.open(path).convert("L"))
    return (m > 0).astype(np.uint8)

def mask_coverage(mask_path: Path) -> float:
    m = load_mask_bin(mask_path)
    return float(m.mean())  # fraction of pixels that are lesion

def image_hw(path: Path) -> tuple[int, int]:
    with Image.open(path) as im:
        w, h = im.size
    return h, w  # (H, W)

# ISIC
isic_img_stems  = stems(ISIC_IMG_DIR, ".jpg")
isic_mask_stems = stems(ISIC_MASK_DIR, ".png")

isic_pairs = isic_img_stems & isic_mask_stems
isic_img_only  = sorted(isic_img_stems - isic_mask_stems)
isic_mask_only = sorted(isic_mask_stems - isic_img_stems)

# HAM
ham_img_stems  = stems(HAM_IMG_DIR, ".jpg")
ham_mask_stems = stems(HAM_MASK_DIR, ".png")

ham_pairs = ham_img_stems & ham_mask_stems
ham_img_only  = sorted(ham_img_stems - ham_mask_stems)
ham_mask_only = sorted(ham_mask_stems - ham_img_stems)

summary = pd.DataFrame([
    {"dataset":"ISIC 2018", "images":len(isic_img_stems), "masks":len(isic_mask_stems), "matched_pairs":len(isic_pairs),
     "image_only":len(isic_img_only), "mask_only":len(isic_mask_only)},
    {"dataset":"HAM10000", "images":len(ham_img_stems), "masks":len(ham_mask_stems), "matched_pairs":len(ham_pairs),
     "image_only":len(ham_img_only), "mask_only":len(ham_mask_only)},
])

summary

## 3-  Resolution stats for HAM10000 and ISIC 2018


In [ ]:
def resolution_df(img_dir: Path, suffix: str) -> pd.DataFrame:
    paths = list_files(img_dir, suffix)

    rows = []
    for p in paths:
        h, w = image_hw(p)
        rows.append({"stem": p.stem, "height": h, "width": w, "pixels": h*w, "aspect_ratio": w / h})
    return pd.DataFrame(rows)

isic_res = resolution_df(ISIC_IMG_DIR, ".jpg")
ham_res  = resolution_df(HAM_IMG_DIR,  ".jpg")

print(f"ISIC2018:\n {isic_res[["height","width","pixels", "aspect_ratio"]].describe()}")
print(f"HAM10000:\n {ham_res[["height","width","pixels", "aspect_ratio"]].describe()}")

# pixel counts distribution
plt.figure()
plt.hist(isic_res["pixels"], bins=30)
plt.title("ISIC 2018 image pixel count distribution")
plt.xlabel("pixels (H×W)")
plt.ylabel("count")
plt.tight_layout()
plt.show()

plt.figure()
plt.hist(ham_res["pixels"], bins=30)
plt.title("HAM10000 image pixel count distribution")
plt.xlabel("pixels (H×W)")
plt.ylabel("count")
plt.tight_layout()
plt.show()

# aspect-ratio counts
# ISIC
plt.figure()
plt.hist(isic_res["aspect_ratio"], bins=30)
plt.title("ISIC 2018 aspect ratio distribution")
plt.xlabel("aspect ratio (width / height)")
plt.ylabel("image count")
plt.tight_layout()
plt.show()

# HAM
plt.figure()
plt.hist(ham_res["aspect_ratio"], bins=5)
plt.title("HAM10000 aspect ratio distribution")
plt.xlabel("aspect ratio (width / height)")
plt.ylabel("image count")
plt.tight_layout()
plt.show()


In [ ]:
# ISIC: round ratios to avoid floating-point noise
isic_ratio_counts = (
    isic_res["aspect_ratio"]
    .round(2)
    .value_counts()
    .sort_index()
)

isic_ratio_counts

dominant = isic_ratio_counts[isic_ratio_counts >= 50]

plt.figure(figsize=(6, 6))
plt.barh(dominant.index.astype(str), dominant.values)
plt.xlabel("image count")
plt.ylabel("aspect ratio (width / height)")
plt.title("ISIC 2018 dominant aspect ratios (≥50 images)")
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()
